In [ ]:
# ============================================================
# BME 507 - WARM-UP EXERCISE 3
# Gender Classification: ResNet-50 vs Swin-Base + Grad-CAM
#
# Before running:
# 1. Colab -> Runtime -> Change runtime type -> GPU
# 2. Upload Gender01.zip to /content/Gender01.zip
# ============================================================

!pip -q install grad-cam

import os
import zipfile
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import (
    resnet50, ResNet50_Weights,
    swin_b, Swin_B_Weights
)

from sklearn.metrics import accuracy_score, roc_auc_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

GENDER_ZIP = "/content/Gender01.zip"
EXTRACT_DIR = "/content/exercise3"

EPOCHS = 5          # Short Colab test. Increase later on Sol.
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. UNZIP DATASET
# ------------------------------------------------------------

if not os.path.exists(GENDER_ZIP):
    raise FileNotFoundError(
        f"{GENDER_ZIP} not found. Upload Gender01.zip to Colab first."
    )

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(GENDER_ZIP, "r") as z:
        z.extractall(EXTRACT_DIR)

train_dir = os.path.join(EXTRACT_DIR, "Gender01", "train")
test_dir  = os.path.join(EXTRACT_DIR, "Gender01", "test")

print("Train directory:", train_dir)
print("Test directory :", test_dir)


# ------------------------------------------------------------
# 3. PREPROCESSING
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

train_dataset = ImageFolder(train_dir, transform=transform)
test_dataset  = ImageFolder(test_dir, transform=transform)

print("\nClasses:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)


# ------------------------------------------------------------
# 5. TRAINING FUNCTION
# ------------------------------------------------------------

def train_model(model, model_name):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE
    )

    print(f"\n{'='*60}")
    print("TRAINING:", model_name)
    print(f"{'='*60}")

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / len(train_dataset)
        epoch_accuracy = correct / total

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Train Accuracy: {epoch_accuracy:.4f}"
        )

    return model


# ------------------------------------------------------------
# 6. TEST FUNCTION
# ------------------------------------------------------------

def evaluate_model(model, model_name):

    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            probabilities = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(labels.numpy())
            all_predictions.extend(predictions.cpu().numpy())

            # Probability of class index 1
            all_probabilities.extend(
                probabilities[:, 1].cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    print(f"\n{model_name}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test AUC:      {auc:.4f}")

    return accuracy, auc


# ------------------------------------------------------------
# 7. RESNET-50
# ------------------------------------------------------------

resnet = resnet50(
    weights=ResNet50_Weights.DEFAULT
)

resnet.fc = nn.Linear(
    resnet.fc.in_features,
    2
)

resnet = train_model(
    resnet,
    "ResNet-50"
)

resnet_accuracy, resnet_auc = evaluate_model(
    resnet,
    "ResNet-50"
)


# Move ResNet to CPU temporarily to free GPU memory
resnet = resnet.cpu()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 8. SWIN-BASE
# ------------------------------------------------------------

swin = swin_b(
    weights=Swin_B_Weights.DEFAULT
)

swin.head = nn.Linear(
    swin.head.in_features,
    2
)

swin = train_model(
    swin,
    "Swin-Base"
)

swin_accuracy, swin_auc = evaluate_model(
    swin,
    "Swin-Base"
)


# ------------------------------------------------------------
# 9. RESULT COMPARISON
# ------------------------------------------------------------

print("\n")
print("=" * 55)
print("FINAL TEST RESULTS")
print("=" * 55)
print(f"{'Model':<15} {'Accuracy':<15} {'AUC':<15}")
print("-" * 55)
print(f"{'ResNet-50':<15} {resnet_accuracy:<15.4f} {resnet_auc:<15.4f}")
print(f"{'Swin-Base':<15} {swin_accuracy:<15.4f} {swin_auc:<15.4f}")


# ------------------------------------------------------------
# 10. GRAD-CAM
# ------------------------------------------------------------

# Grad-CAM shows image regions that influenced the model prediction.

def swin_reshape_transform(tensor):
    # torchvision Swin produces:
    # Batch x Height x Width x Channels
    return tensor.permute(0, 3, 1, 2)


def show_gender_cam(model, model_name, sample_index):

    model = model.to(device)
    model.eval()

    image_path, true_label = test_dataset.samples[sample_index]

    original_image = Image.open(image_path).convert("RGB")
    display_image = original_image.resize((224, 224))

    rgb_image = np.array(display_image).astype(np.float32) / 255.0

    input_tensor = transform(original_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        predicted_class = output.argmax(dim=1).item()

    true_name = test_dataset.classes[true_label]
    predicted_name = test_dataset.classes[predicted_class]

    if model_name == "ResNet-50":

        target_layers = [model.layer4[-1]]

        cam = GradCAM(
            model=model,
            target_layers=target_layers
        )

    else:

        # Last Swin transformer block
        target_layers = [
            model.features[-1][-1].norm1
        ]

        cam = GradCAM(
            model=model,
            target_layers=target_layers,
            reshape_transform=swin_reshape_transform
        )

    targets = [
        ClassifierOutputTarget(predicted_class)
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(rgb_image)
    plt.title(
        f"Original\nTrue: {true_name}"
    )
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.title(
        f"{model_name} Grad-CAM\nPrediction: {predicted_name}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    model = model.cpu()
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 11. FIND ONE FEMALE AND ONE MALE TEST IMAGE
# ------------------------------------------------------------

female_class = train_dataset.class_to_idx["female"]
male_class   = train_dataset.class_to_idx["male"]

female_index = next(
    i for i, (_, label) in enumerate(test_dataset.samples)
    if label == female_class
)

male_index = next(
    i for i, (_, label) in enumerate(test_dataset.samples)
    if label == male_class
)


# ------------------------------------------------------------
# 12. SHOW CAM RESULTS
# ------------------------------------------------------------

# ResNet
show_gender_cam(
    resnet,
    "ResNet-50",
    female_index
)

show_gender_cam(
    resnet,
    "ResNet-50",
    male_index
)

# Swin
show_gender_cam(
    swin,
    "Swin-Base",
    female_index
)

show_gender_cam(
    swin,
    "Swin-Base",
    male_index
)


# ------------------------------------------------------------
# 13. SAVE MODELS
# ------------------------------------------------------------

torch.save(
    resnet.state_dict(),
    "/content/resnet50_gender.pth"
)

torch.save(
    swin.state_dict(),
    "/content/swin_base_gender.pth"
)

print("\nDone.")
print("Saved:")
print("/content/resnet50_gender.pth")
print("/content/swin_base_gender.pth")

In [ ]:
import os
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Folder where everything will be saved
SAVE_DIR = "/content/drive/MyDrive/BME507/Warmup_3_4"
os.makedirs(SAVE_DIR, exist_ok=True)

# Exercise 3 results from your completed run
exercise3_results = pd.DataFrame({
    "Model": ["ResNet-50", "Swin-Base"],
    "Train_Accuracy": [0.9935, 0.9286],
    "Test_Accuracy": [0.8387, 0.8602],
    "Test_AUC": [0.9461, 0.9251],
    "Epochs": [3, 3]
})

exercise3_results.to_csv(
    os.path.join(SAVE_DIR, "exercise3_gender_results.csv"),
    index=False
)

print(exercise3_results)
print("\nSaved to:")
print(os.path.join(SAVE_DIR, "exercise3_gender_results.csv"))